In [ ]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from instruments.gilson.gilson_ethernet import GilsonEthernet
from instruments.gilson.tray import Tray
from instruments.gilson.rack import Rack_209, Rack_3dp
from instruments.gilson.probe import ProbeState    # NEW
from core.logging import flow_logger as logger
from instruments.vici.dim import DIM
from instruments.vici.fsm import FSM
from instruments.syr_pumps.HB_syr_pump import HBElite
from instruments.knauer.knauer_pump_azura import KnauerPumpAzura
from ecosystems.reaction_segment_generation import RSG
from ecosystems.segmentation import Segmentation


# Experiment framework
from core.experiment_context import ExperimentManager
from core.experiment_builder import ExperimentBuilder
from core.experiment_validator import ExperimentValidator
from core.experiment_compiler import ExperimentCompiler
from core.experiment_intent import ExperimentIntent
from core.inventory import Inventory

In [ ]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

In [ ]:
# --- Instantiate the tray ---
tray = Tray()

# --- Instantiate modules for the tray ---
rack1 = Rack_209()
rack2 = Rack_3dp()

# --- Assign modules to tray slots ---
tray.assign_slot(1, rack1, alias = "rack1")    # Assigns rack1 to slot 1
tray.assign_slot(2, rack2, alias = "rack2")    # Assigns rack2 to slot 2
tray.assign_slot(3, dim, alias = "dim")      # Assigns dim to slot 3

In [ ]:
# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Instantiate the probe (state machine only) ---
probe_state = ProbeState()

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1E-9")

In [ ]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, DIM and probe
rsg = RSG(gilson=g, pump=syr_pump, dim=dim, probe_state=probe_state)

# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)